# 02a — Validate Label Survey

**PURPOSE**: Validate the ingested prototype label-survey artifact (mbn/life,
2026-01-01..2026-08-07, 225 articles) for row integrity — uniqueness, null
rates, period boundaries, duplicates — and freeze it into the canonical
`article_index` table. This notebook does **not** normalize bracket labels
(see 02b) and does **not** collect article bodies (see 03).

**INPUT**: `data/00_raw/mbn/life/index/mbn_life_label_survey.source.json`
(frozen copy of `crawl/mbn_label_survey.py`'s output; that script and its
original run live outside this repository — see `config/pipeline.yaml:source_survey`
for provenance).

**OUTPUT**:
- `data/20_processed/mbn/life/article_index.{parquet,csv}` (225 rows, PK=`articleId`)
- `data/80_quality/label_survey_quality_report.json`

**DEPENDENCIES**: pandas, pyarrow, pyyaml, `src.parsing.mbn_index_parser`,
`src.validation.checks`, `src.io.*`

**PARAMETERS**: read from `config/pipeline.yaml` (`period.start`, `period.end`,
`source_survey.*`, `versions.index_parser`) — no magic numbers inline.

**ASSUMPTIONS**: the source JSON has keys
`{period,total_articles,labeled_count,unlabeled_count,unique_label_count,label_frequency,articles}`
with each `articles[i]` = `{category,id,date,label,title,url}`; the prototype
crawler already deduplicated by `articleId` within its own run.

**SIDE EFFECTS**: writes the two output artifacts above under `data/`; no
network calls; does not modify `data/00_raw/**` (read-only input).

**FAIL CONDITIONS**: raises `PrimaryKeyViolation` (hard stop) if `articleId`
is null or duplicated in the source — that would silently corrupt every
downstream join. All other checks (row-count match, boundary dates, page-12
boundary count, duplicate URL/(date,title)) are recorded in the quality
report as PASS/WARN rather than raised, since they are informative, not
join-breaking.

In [1]:
# --- bootstrap: locate repo root and put it on sys.path (no hardcoded absolute paths) ---
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (.git marker) from {start}")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE


In [2]:
import json
from datetime import datetime, timezone

import pandas as pd
import yaml

from src.io.hashing import sha256_file
from src.io.paths import config_path, data_dir, ensure_parent
from src.parsing.mbn_index_parser import (
    INDEX_PARSER_VERSION,
    survey_articles_to_index_records,
)
from src.validation.checks import (
    PrimaryKeyViolation,
    assert_primary_key,
    duplicate_report,
    null_counts,
)

with open(config_path(REPO_ROOT, "pipeline.yaml"), encoding="utf-8") as f:
    PIPELINE_CFG = yaml.safe_load(f)

PERIOD_START = PIPELINE_CFG["period"]["start"]
PERIOD_END = PIPELINE_CFG["period"]["end"]
SOURCE_CFG = PIPELINE_CFG["source_survey"]

INPUT_PATH = REPO_ROOT / SOURCE_CFG["relative_path"]
OUTPUT_INDEX_PATH = data_dir(REPO_ROOT, "20_processed", "mbn", "life", "article_index")
OUTPUT_QUALITY_PATH = data_dir(REPO_ROOT, "80_quality", "label_survey_quality_report.json")

EXECUTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()

print("INPUT_PATH:", INPUT_PATH)
print("PERIOD:", PERIOD_START, "..", PERIOD_END)

INPUT_PATH: /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/00_raw/mbn/life/index/mbn_life_label_survey.source.json
PERIOD: 2026-01-01 00:00 .. 2026-08-07 23:59


## Load source survey and compute input hash

In [3]:
INPUT_SHA256 = sha256_file(INPUT_PATH)

with open(INPUT_PATH, encoding="utf-8") as f:
    survey = json.load(f)

print("input sha256:", INPUT_SHA256)
print("survey top-level keys:", list(survey.keys()))
print("survey.total_articles:", survey["total_articles"])
print("survey.labeled_count:", survey["labeled_count"])
print("survey.unlabeled_count:", survey["unlabeled_count"])
print("survey.unique_label_count:", survey["unique_label_count"])

input sha256: b76ecb2a26a69bba8f963ab44b39e229802ce9ea0cda182413cedd065a123c5a
survey top-level keys: ['period', 'total_articles', 'labeled_count', 'unlabeled_count', 'unique_label_count', 'label_frequency', 'articles']
survey.total_articles: 225
survey.labeled_count: 102
survey.unlabeled_count: 123
survey.unique_label_count: 15


## Build article_index records and enforce primary-key integrity (hard stop on violation)

In [4]:
records = survey_articles_to_index_records(
    survey,
    source=PIPELINE_CFG["source"],
    section=PIPELINE_CFG["section"],
    collected_at=EXECUTION_TIMESTAMP,
    raw_sha256=INPUT_SHA256,
)
article_index = pd.DataFrame.from_records(records)

# Hard stop: articleId must be unique and non-null before anything downstream
# is allowed to join against this table.
assert_primary_key(article_index, ["articleId"], context="article_index")
print("PK check passed: articleId is unique and non-null across", len(article_index), "rows")
article_index.head(3)

PK check passed: articleId is unique and non-null across 225 rows


,articleId,source,sourceArticleId,url,rawTitle,publishedAt,section,collectedAt,rawSha256,indexParserVersion,hasBracketLabel
0,5210942,mbn,5210942,https://www.mbn.co.kr/news/life/5210942,[AI기상캐스터]'입추'에도 서울 폭염 절정 한낮39도…주말 동해안 중심 비,2026-08-07 13:04,life,2026-08-07T11:24:25.357280+00:00,b76ecb2a26a69bba8f963ab44b39e229802ce9ea0cda18...,mbn_index_parser@1,True
1,5210780,mbn,5210780,https://www.mbn.co.kr/news/life/5210780,"""달리며 탄소중립 실천""…'제12회 I LOVE 방송대 마라톤 축제' 9월 5일 개최",2026-08-06 18:43,life,2026-08-07T11:24:25.357280+00:00,b76ecb2a26a69bba8f963ab44b39e229802ce9ea0cda18...,mbn_index_parser@1,False
2,5210752,mbn,5210752,https://www.mbn.co.kr/news/life/5210752,[Season Item] 호텔이 여름 시즌을 준비하는 방법…올해 여름 제철 미식 트...,2026-08-06 17:14,life,2026-08-07T11:24:25.357280+00:00,b76ecb2a26a69bba8f963ab44b39e229802ce9ea0cda18...,mbn_index_parser@1,True


## Row-count / null / period-boundary / page-12-boundary / duplicate checks (recorded, not raised)

In [5]:
checks = {}

checks["row_count"] = int(len(article_index))
checks["expected_row_count"] = SOURCE_CFG["expected_total_articles"]
checks["row_count_matches_expected"] = checks["row_count"] == checks["expected_row_count"]

checks["labeled_count"] = int(article_index["hasBracketLabel"].sum())
checks["unlabeled_count"] = int((~article_index["hasBracketLabel"]).sum())
checks["expected_labeled_count"] = SOURCE_CFG["expected_labeled_articles"]
checks["expected_unlabeled_count"] = SOURCE_CFG["expected_unlabeled_articles"]
checks["labeled_count_matches_expected"] = checks["labeled_count"] == checks["expected_labeled_count"]
checks["unlabeled_count_matches_expected"] = checks["unlabeled_count"] == checks["expected_unlabeled_count"]

checks["null_counts"] = null_counts(article_index)

min_date, max_date = article_index["publishedAt"].min(), article_index["publishedAt"].max()
checks["min_published_at"] = min_date
checks["max_published_at"] = max_date
checks["min_date_within_period_start"] = min_date >= PERIOD_START
checks["max_date_within_period_end"] = max_date <= PERIOD_END

# Page-12 boundary: the original crawl log recorded 20 fetched / 5 newly
# in-range on page 12, bounded by page 11's oldest item (2026-01-07 18:30).
# The frozen survey has no per-page column, so this replays the same window
# on publishedAt directly.
PAGE12_BOUNDARY_UPPER = "2026-01-07 18:30"
boundary_mask = (article_index["publishedAt"] >= PERIOD_START) & (article_index["publishedAt"] < PAGE12_BOUNDARY_UPPER)
checks["page12_boundary_window"] = [PERIOD_START, PAGE12_BOUNDARY_UPPER]
checks["page12_boundary_article_count"] = int(boundary_mask.sum())
checks["page12_boundary_count_is_5"] = checks["page12_boundary_article_count"] == 5

checks["duplicate_counts"] = duplicate_report(
    article_index, [["articleId"], ["url"], ["publishedAt", "rawTitle"]]
)

for k, v in checks.items():
    print(f"{k}: {v}")

row_count: 225
expected_row_count: 225
row_count_matches_expected: True
labeled_count: 102
unlabeled_count: 123
expected_labeled_count: 102
expected_unlabeled_count: 123
labeled_count_matches_expected: True
unlabeled_count_matches_expected: True
null_counts: {'articleId': 0, 'source': 0, 'sourceArticleId': 0, 'url': 0, 'rawTitle': 0, 'publishedAt': 0, 'section': 0, 'collectedAt': 0, 'rawSha256': 0, 'indexParserVersion': 0, 'hasBracketLabel': 0}
min_published_at: 2026-01-01 17:23
max_published_at: 2026-08-07 13:04
min_date_within_period_start: True
max_date_within_period_end: True
page12_boundary_window: ['2026-01-01 00:00', '2026-01-07 18:30']
page12_boundary_article_count: 5
page12_boundary_count_is_5: True
duplicate_counts: {'articleId': 0, 'url': 0, 'publishedAt+rawTitle': 0}


## Recompute raw label frequency and labeled-article count directly from titles (cross-check vs. survey's own summary)

In [6]:
from src.parsing.mbn_index_parser import extract_bracket_label

recomputed_labels = article_index["rawTitle"].map(extract_bracket_label)
recomputed_label_counts = recomputed_labels.value_counts(dropna=True).to_dict()
recomputed_labeled_count = int(recomputed_labels.notna().sum())

checks["recomputed_labeled_count"] = recomputed_labeled_count
checks["recomputed_labeled_count_matches_survey"] = recomputed_labeled_count == survey["labeled_count"]
checks["recomputed_unique_raw_label_count"] = len(recomputed_label_counts)
checks["recomputed_label_frequency"] = recomputed_label_counts

print("recomputed labeled count:", recomputed_labeled_count, "(survey says:", survey["labeled_count"], ")")
print("recomputed unique raw labels:", len(recomputed_label_counts), "(survey says:", survey["unique_label_count"], ")")

recomputed labeled count: 102 (survey says: 102 )
recomputed unique raw labels: 15 (survey says: 15 )


## Determine quality verdict and write outputs

In [7]:
blocking_issues = []
if not checks["row_count_matches_expected"]:
    blocking_issues.append("row_count mismatch")
if any(v > 0 for v in checks["null_counts"].values()):
    blocking_issues.append("unexpected nulls in article_index")

warnings = []
if not checks["page12_boundary_count_is_5"]:
    warnings.append("page12 boundary window did not contain exactly 5 articles")
if not checks["recomputed_labeled_count_matches_survey"]:
    warnings.append("recomputed labeled count does not match survey summary")
if any(v > 0 for v in checks["duplicate_counts"].values()):
    warnings.append("duplicate rows detected (see duplicate_counts)")

quality_status = "FAIL" if blocking_issues else ("WARN" if warnings else "PASS")

from src.io.parquet_io import write_table

write_manifest = write_table(article_index, OUTPUT_INDEX_PATH, required_columns=["articleId", "rawTitle", "publishedAt"])

quality_report = {
    "notebook": "02aValidateLabelSurvey.ipynb",
    "executionTimestamp": EXECUTION_TIMESTAMP,
    "inputPath": str(INPUT_PATH.relative_to(REPO_ROOT)),
    "inputSha256": INPUT_SHA256,
    "indexParserVersion": INDEX_PARSER_VERSION,
    "checks": checks,
    "blockingIssues": blocking_issues,
    "warnings": warnings,
    "qualityStatus": quality_status,
    "outputArtifact": write_manifest,
}

ensure_parent(OUTPUT_QUALITY_PATH)
with open(OUTPUT_QUALITY_PATH, "w", encoding="utf-8") as f:
    json.dump(quality_report, f, ensure_ascii=False, indent=2, default=str)

print("qualityStatus:", quality_status)
print("blockingIssues:", blocking_issues)
print("warnings:", warnings)
print("wrote:", OUTPUT_QUALITY_PATH.relative_to(REPO_ROOT))

qualityStatus: PASS
blockingIssues: []
warnings: []
wrote: data/80_quality/label_survey_quality_report.json


## Closing summary

In [8]:
from src.io.hashing import sha256_file as _sha

print("=== ROW COUNTS ===")
print({"article_index": len(article_index)})

print("=== NULL COUNTS ===")
print(checks["null_counts"])

print("=== DUPLICATES ===")
print(checks["duplicate_counts"])

print("=== QUALITY METRICS ===")
print({
    "row_count_matches_expected": checks["row_count_matches_expected"],
    "labeled_count_matches_expected": checks["labeled_count_matches_expected"],
    "page12_boundary_count_is_5": checks["page12_boundary_count_is_5"],
    "recomputed_labeled_count_matches_survey": checks["recomputed_labeled_count_matches_survey"],
})

print("=== OUTPUT PATH ===")
print(write_manifest["parquet_path"], "|", write_manifest["csv_path"], "|", str(OUTPUT_QUALITY_PATH))

print("=== OUTPUT HASH ===")
print({"parquet_sha256": write_manifest["parquet_sha256"], "csv_sha256": write_manifest["csv_sha256"]})

print("=== NEXT NOTEBOOK ===")
print("02bNormalizeEditorialLabels.ipynb")

=== ROW COUNTS ===
{'article_index': 225}
=== NULL COUNTS ===
{'articleId': 0, 'source': 0, 'sourceArticleId': 0, 'url': 0, 'rawTitle': 0, 'publishedAt': 0, 'section': 0, 'collectedAt': 0, 'rawSha256': 0, 'indexParserVersion': 0, 'hasBracketLabel': 0}
=== DUPLICATES ===
{'articleId': 0, 'url': 0, 'publishedAt+rawTitle': 0}
=== QUALITY METRICS ===
{'row_count_matches_expected': True, 'labeled_count_matches_expected': True, 'page12_boundary_count_is_5': True, 'recomputed_labeled_count_matches_survey': True}
=== OUTPUT PATH ===
/home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/article_index.parquet | /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/20_processed/mbn/life/article_index.csv | /home/sieg/projects-wsl/mbN_GUIDE_PY/mbN_GUIDE/data/80_quality/label_survey_quality_report.json
=== OUTPUT HASH ===
{'parquet_sha256': '19515b8133b445dace22ec955e78b5f3903c08768c2d3779272b0bce4aac54f5', 'csv_sha256': '4444b064b468563a44e1cb869543e521fbda6224aeda869d6b27f3b481a

## 라벨 사용 패턴 EDA 시각화

검증된 `article_index`를 사용해 **대괄호 라벨 존재 여부**를 이진 타깃으로 탐색한다. 제목 길이·발행 시간·월·요일·반복 라벨과의 관계를 기술하며, 이는 기사 품질이나 인과관계를 뜻하지 않는다.

In [ ]:
from IPython.display import Image, Markdown, display

from src.analysis.label_survey_eda import build_label_survey_eda

EDA_OUTPUT_DIR = REPO_ROOT / "outputs" / "eda_label_survey"
eda_manifest = build_label_survey_eda(article_index, EDA_OUTPUT_DIR)

display(Markdown(
    f"**Target 설계:** `hasBracketLabel` (이진)  \\n"
    f"**관찰 범위:** MBN Life 동결 표본 {eda_manifest['rows']:,}건, 라벨 있음 {eda_manifest['labeled_rows']:,}건 ({eda_manifest['label_rate']:.1%})  \\n"
    "**원인 가설:** 시간·제목 형식·반복 편집 라벨의 동시 패턴을 탐색한다.  \\n"
    "**제한:** 탐색적 기술통계이며 기사 품질 또는 인과 증거가 아니다.  \\n"
    "**결론:** CSV·PNG 산출물을 재실행 가능한 형태로 저장했다."
))

for figure_name, figure_path in eda_manifest['figures'].items():
    display(Markdown(f"### {figure_name}"))
    display(Image(filename=figure_path))

print("EDA output:", eda_manifest['output_dir'])
print("CSV artifacts:", *eda_manifest['tables'].values(), sep="\n- ")